# Vectorizing `VeriGym` environments

In this tutorial, we showcase how to create vectorized versions of `VeriGym` environments.
We cover the three main types of instantiable `VeriGymEnv`s: `GenerativeEnv`, `ExplicitEnv`, and `FrameworkExplicitEnv`.
There are also three ways of vectorizing environments: using `gymnasium`-native functions, using the interface from `stable_baselines3`, and using `VeriGym`-native functions.

### How can we vectorize different types of `VeriGymEnv`s?

The table below summarizes which combinations of API and `VeriGymEnv` work.

| | `gymnasium.make_vec` | `sb3.make_vec_env` | `VeriGymEnv.make_vec` | Notes |
|---|---|---|---|---|
|`GenerativeEnv` | ❌ | ✅ | ✅ | As of now, generative envs can only be instantiated from a `gymnasium.Env`, but `gymnasium.make_vec` requires an `id`. `GenerativeEnv` is not registered in `gymnasium`.|
|`ExplicitEnv` | ✅ | ✅ | ✅ | |
| `FrameworkExplicitEnv` | (✅) | (✅) | (✅) | For envs built from `stormpy`,  only `vectorization_mode="sync"` is available due to serialization issues stormpy MDP objects.| 

### Why do we want to vectorize?

Vectorized environments allow for parallel training on several environments at the same time - this can drastically increase the data throughput and decrease the training time.


## Imports

In [5]:
from verigym.environments.generativeenv import GenerativeEnv
from verigym.environments.explicitenv import ExplicitEnv
from verigym.environments.frameworkexplicitenv import FrameworkExplicitEnv
from verigym.environments.transition_func import TransitionFunction
from verigym.environments.reward_func import RewardFunction
from verigym.frameworks.stormpy.stormpy_utils import load_stormpy_model
from verigym.frameworks.stormpy.formatter import StormpyFormatter

import gymnasium as gym
import numpy as np
from stable_baselines3.common.env_util import make_vec_env

## Vectorizing `GenerativeEnvs`

In [6]:
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")

vec_gen_env = GenerativeEnv.vec_from_gymnasium(env,
                                               num_envs=4,
                                               vectorization_mode="sync")

vec_gen_env = gym.make_vec("FrozenLake-v1", is_slippery=False, render_mode="rgb_array", num_envs=4)


In [7]:
dir(env)
print(env.spec)

EnvSpec(id='FrozenLake-v1', entry_point='gymnasium.envs.toy_text.frozen_lake:FrozenLakeEnv', reward_threshold=0.7, nondeterministic=False, max_episode_steps=100, order_enforce=True, disable_env_checker=False, kwargs={'map_name': '4x4', 'is_slippery': False, 'render_mode': 'rgb_array'}, namespace=None, name='FrozenLake', version=1, additional_wrappers=(), vector_entry_point=None)


In [8]:
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
fig, axs = plt.subplots(nrows=2, ncols=2)

vec_gen_env.reset()
for _ in range(10):
    actions = vec_gen_env.action_space.sample()
    print(actions)
    obs, rew, term, trunc, info = vec_gen_env.step(actions)
    print(obs)
    frame = [env.render() for env in vec_gen_env.envs]

    clear_output(wait=True)
    for i in range(4):
        axs[i%2, i//2].imshow(frame[i])
    display(fig)
    plt.close()


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def call_make():
    env = gym.make("FrozenLake-v1", is_slippery=False)
    return GenerativeEnv.from_gymnasium(env)
sb3_vec_env = make_vec_env(call_make, n_envs=4)

## Vectorizing `ExplicitEnvs`

In [ ]:
import graphviz


g_desc = \
"""
digraph MDP {
    layout=neato
    s0a [xlabel="a", shape=point, pos="2,0!"]
    s0b [xlabel="b", shape=point, pos="0.5,1!"]
    s1a [xlabel="a", shape=point, pos="1,0.25!"]
    s1b [xlabel="b", shape=point, pos="3,0.75!"]

    s0 [label="s0", pos="0,0!"]
    s1 [label="s1", pos="2,1!"]
    s2 [label="s2", pos="4,0!"]

    s0 -> s0a [arrowhead=none]
    s0 -> s0b [arrowhead=none]
    s1 -> s1a [arrowhead=none]
    s1 -> s1b [arrowhead=none]
    
    s0a -> s1 [label="0.5"]
    s0a -> s2 [label="0.5"]
    s0b -> s0 [label="0.2"]
    s0b -> s1 [label="0.8"]

    s1a -> s0 [label="0.5"]
    s1a -> s1 [label="0.5"]
    s1b -> s1 [label="0.2"]
    s1b -> s2 [label="0.8"]
}
"""
g = graphviz.Digraph("MDP", g_desc)

g

In [ ]:
num_envs = 4
T_array = np.array(
        [[[0.0, 0.5, 0.5], # s0, a
        [0.2, 0.8, 0.0]], # s0, b
        [[0.5, 0.5, 0.0], # s1, a
            [0.0, 0.2, 0.8]],# s1, b
            [[0.0, 0.0, 1.0],
            [0.0, 0.0, 1.0]]]
    )
    
T = TransitionFunction.from_array(
    T_array
)
R = RewardFunction.from_array(
    np.array(
        [[1.0, 0.0],
        [0.2, 1.0],
        [0.0, 0.0]]
    )
)
kwargs = {
    "nr_states": 3,
    "nr_actions": 2,
    "initial_state_distr": np.array([1.0, 0.0, 0.0]),
    "transition_function": T,
    "reward_function": R
}
vec_env_sync = ExplicitEnv.make_vec(num_envs, vectorization_mode="sync",
                                    **kwargs)
vec_env_async = ExplicitEnv.make_vec(num_envs=num_envs, vectorization_mode="async",
                                        **kwargs)


In [ ]:
gym.make_vec("ExplicitEnv-v0", num_envs=4, vectorization_mode="sync", **kwargs)
gym.make_vec("ExplicitEnv-v0", num_envs=4, vectorization_mode="async", **kwargs)

AsyncVectorEnv(ExplicitEnv-v0, num_envs=4)

In [ ]:
def call_make_explicit():
        return ExplicitEnv(**kwargs)
make_vec_env("ExplicitEnv-v0", env_kwargs=kwargs, n_envs=4)

/Users/juleschmidt/Documents/VeriGym/.venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:729: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes ([]).
  logger.warn(


## Vectorizing `FrameworkExplicitEnv`s

In [ ]:
mdp = load_stormpy_model("../tests/test_2d.prism")
fw_env = FrameworkExplicitEnv.vec_from_stormpy(mdp, num_envs=4, vectorization_mode="sync")

In [ ]:
try:
    FrameworkExplicitEnv.vec_from_stormpy(mdp, num_envs=4, vectorization_mode="async")
except ValueError as e:
    print("Value Error: ", e)

Value Error:  Cannot use async vectorization with built stormpy MDP (C++ object that cannot be serialized).


In [ ]:
kwargs = {"model": mdp, "formatter": StormpyFormatter(mdp)}
gym.make_vec("FrameworkExplicitEnv-v0", num_envs=1, vectorization_mode="sync", **kwargs)

SyncVectorEnv(FrameworkExplicitEnv-v0, num_envs=1)

In [ ]:
try:
    gym.make_vec("FrameworkExplicitEnv-v0", num_envs=4, vectorization_mode="async", **kwargs)
except TypeError as e:
    print("TypeError: ", e)

TypeError:  cannot pickle 'stormpy.storage._storage.SparseMdp' object


In [ ]:
make_vec_env("FrameworkExplicitEnv-v0", env_kwargs=kwargs, n_envs=4)